In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================
# TRAFFIC PREDICTION PIPELINE
# ============================================

# ============================================
# ENVIRONMENT
# ============================================

%run /content/drive/MyDrive/project_cd/config/environment.py

# ============================================
# IMPORTS
# ============================================

import pandas as pd
import os

from config.paths import PATHS

from domains.traffic.modules.models.congestion_model import (
    train_congestion_model,
    generate_predictions,
    evaluate_model,
    classify_risk
)

# ============================================
# LOAD FEATURE DATA
# ============================================

features_file_path = PATHS["traffic"]["features"] + "traffic_features.csv"

# Ensure the features directory exists
os.makedirs(PATHS["traffic"]["features"], exist_ok=True)

try:
    if not os.path.exists(features_file_path):
        raise FileNotFoundError(f"The file '{features_file_path}' does not exist.")

    if os.stat(features_file_path).st_size == 0:
        raise pd.errors.EmptyDataError(f"The file '{features_file_path}' is empty.")

    df = pd.read_csv(features_file_path)

    print("Traffic feature dataset loaded ✅")
    print(df.head())

    if df.empty:
        print(f"Warning: DataFrame loaded from '{features_file_path}' is empty despite file not being empty. Please check CSV content for data rows.")

except FileNotFoundError as e:
    print(f"Error: {e}")
    print(f"Please ensure your 'traffic_features.csv' with real data is located at: {features_file_path}")
    df = pd.DataFrame() # Create an empty DataFrame to prevent further errors
except pd.errors.EmptyDataError as e:
    print(f"Error: {e}")
    print(f"The file '{features_file_path}' is empty. Please ensure it contains data.")
    df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading the data from '{features_file_path}': {e}")
    df = pd.DataFrame()


# ============================================
# SELECT MODEL FEATURES
# ============================================

# Only proceed if the DataFrame is not empty
if not df.empty:
    feature_columns = [
        "vehicle_count",
        "crowd_density",
        "traffic_volume",
        "crowd_pressure",
        "vehicle_density_ratio",
        "crowd_vehicle_interaction",
        "high_congestion_flag",
        "day_of_week",
        "is_weekend"
    ]

    X = df[feature_columns]

    # ============================================
    # TARGET VARIABLE
    # ============================================

    y = df["congestion_score"]

    # ============================================
    # TRAIN MODEL
    # ============================================

    model = train_congestion_model(
        X,
        y
    )

    print("Congestion model trained ✅")

    # ============================================
    # GENERATE PREDICTIONS
    # ============================================

    predictions = generate_predictions(
        model,
        X
    )

    df["predicted_congestion_score"] = predictions

    # ============================================
    # GENERATE RISK LEVELS
    # ============================================

    df["predicted_risk_level"] = (

        df["predicted_congestion_score"]
        .apply(classify_risk)
    )

    # ============================================
    # MODEL EVALUATION
    # ============================================

    metrics = evaluate_model(
        y,
        predictions
    )

    print("\nModel Evaluation Metrics")
    print(metrics)

    # ============================================
    # SAVE PREDICTIONS
    # ============================================

    output_path = (

        PATHS["traffic"]["predictions"] +
        "traffic_predictions.csv"
    )

    os.makedirs(PATHS["traffic"]["predictions"], exist_ok=True)
    df.to_csv(
        output_path,
        index=False
    )

    # ============================================
    # OUTPUT
    # ============================================

    print("\nPrediction sample:")
    print(

        df[[

            "route_id",

            "checkpoint_name",

            "congestion_level",

            "predicted_congestion_score",

            "predicted_risk_level"

        ]].head()
    )

    print(f"\nPredictions saved at: {output_path}")

    print("Traffic Prediction Pipeline complete ✅")
else:
    print("Skipping model training and prediction as the feature dataset is empty or could not be loaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment initialized Saurabh✅
Traffic feature dataset loaded ✅
Empty DataFrame
Columns: [calendar_date, route_id, checkpoint_name, vehicle_count, congestion_level, crowd_density, congestion_score, traffic_volume, crowd_pressure, vehicle_density_ratio, crowd_vehicle_interaction, high_congestion_flag, day_of_week, is_weekend]
Index: []


ValueError: Found array with 0 sample(s) (shape=(0, 9)) while a minimum of 1 is required by LinearRegression.

In [ ]:
df

,calendar_date,route_id,checkpoint_name,vehicle_count,congestion_level,crowd_density,congestion_score,traffic_volume,crowd_pressure,vehicle_density_ratio,crowd_vehicle_interaction,high_congestion_flag,day_of_week,is_weekend
